In [1]:
import sympy as sp
from IPython.display import display, Math

# 1. Configuración gráfica
sp.init_printing(use_latex='mathjax')

# 2. Definición de variables
t, w, s = sp.symbols('t w s', real=True)
a = sp.symbols('a', positive=True, real=True)

def transformada_modo_humano(lista_tramos, usar_trigonometria=False):
    print("==================================================")
    print(" PROCEDIMIENTO: TRANSFORMADA (MÉTODO DE EULER)")
    print("==================================================\n")
    
    nucleo = sp.exp(-sp.I * w * t)
    resultado_total = 0
    
    for i, (funcion_t, lim_inf, lim_sup) in enumerate(lista_tramos, 1):
        print(f"---> RESOLVIENDO TRAMO {i} <---")
        
        # 1. Planteamiento Original
        print("\n1. Planteamiento original:")
        display(sp.Integral(funcion_t * nucleo, (t, lim_inf, lim_sup)))
        
        # 2. Identidad de Euler
        funcion_euler = funcion_t.rewrite(sp.exp)
        if funcion_euler != funcion_t:
            print("\n2. Aplicando Identidad de Euler:")
            display(sp.Integral(funcion_euler * nucleo, (t, lim_inf, lim_sup)))
        
        # 3. Distribuyendo
        print("\n3. Distribuyendo y agrupando exponentes:")
        integrando_agrupado = sp.powsimp(sp.expand(funcion_euler * nucleo), combine='exp')
        display(sp.Integral(integrando_agrupado, (t, lim_inf, lim_sup)))
        
        # 4. Antiderivada
        print("\n4. Antiderivada:")
        try:
            antiderivada = sp.integrate(integrando_agrupado, t)
            # FILTRO ANTI-PÁNICO (Para Piecewise)
            if isinstance(antiderivada, sp.Piecewise):
                antiderivada = antiderivada.args[0][0]
            display(antiderivada)
        except:
            print("Integral muy compleja.")
            continue

        # 5. Evaluando límites
        eval_sup = 0 if lim_sup == sp.oo else antiderivada.subs(t, lim_sup)
        eval_inf = 0 if lim_inf == -sp.oo else antiderivada.subs(t, lim_inf)
        
        print(f"\n5. Evaluando límites [ Sup ({lim_sup}) - Inf ({lim_inf}) ]:")
        display(Math(rf"\left[ {sp.latex(eval_sup)} \right] - \left[ {sp.latex(eval_inf)} \right]"))
        
        resultado_total += (eval_sup - eval_inf)
        
    print("\n==================================================")
    print(" 6. RESULTADO FINAL SIMPLIFICADO:")
    
    # TRUCO DE LAPLACE
    try:
        res_s = sp.simplify(resultado_total.subs(w, -sp.I * s))
        res_limpio = res_s.subs(s, sp.I * w)
    except:
        res_limpio = sp.simplify(resultado_total)
        
    # INTERRUPTOR DE SEGURIDAD
    if usar_trigonometria:
        print("(Aplicando retorno a senos y cosenos)")
        res_final = sp.simplify(res_limpio.rewrite(sp.cos))
    else:
        res_final = res_limpio
        
    display(res_final)
    print("==================================================")


# ==========================================
# 📝 ZONA DE CONFIGURACIÓN
# ==========================================

# Tu función aquí
mi_funcion = sp.exp(-(t+4))

mis_tramos = [
    (mi_funcion, -4, sp.oo)
]

# EL INTERRUPTOR DE SEGURIDAD:
# Cambia a True SOLO si tu función original tiene un sen() o cos()
# Cambia a False para problemas con exponenciales normales o polinomios
forzar_cosenos = False

transformada_modo_humano(mis_tramos, usar_trigonometria=forzar_cosenos)

 PROCEDIMIENTO: TRANSFORMADA (MÉTODO DE EULER)

---> RESOLVIENDO TRAMO 1 <---

1. Planteamiento original:


∞                    
⌠                    
⎮   -ⅈ⋅t⋅w  -t - 4   
⎮  ℯ      ⋅ℯ       dt
⌡                    
-4                   


3. Distribuyendo y agrupando exponentes:


∞                    
⌠                    
⎮   -ⅈ⋅t⋅w - t - 4   
⎮  ℯ               dt
⌡                    
-4                   


4. Antiderivada:


               ⅈ               
───────────────────────────────
   4  t  ⅈ⋅t⋅w      4  t  ⅈ⋅t⋅w
w⋅ℯ ⋅ℯ ⋅ℯ      - ⅈ⋅ℯ ⋅ℯ ⋅ℯ     


5. Evaluando límites [ Sup (oo) - Inf (-4) ]:


<IPython.core.display.Math object>


 6. RESULTADO FINAL SIMPLIFICADO:


 4⋅ⅈ⋅w 
ℯ      
───────
ⅈ⋅w + 1

# GUÍA RÁPIDA MA2N: CONFIGURACIÓN DEL SCRIPT DE FOURIER

## 1. El Interruptor de Seguridad: `usar_trigonometria`
Este interruptor (`True` / `False`) controla si el script usa el Método de Euler y si empaqueta la respuesta final en términos de senos y cosenos. **Usarlo mal puede arruinar la notación formal que espera el auxiliar.**

| ¿Qué contiene la función $f(t)$ en el examen? | Valor a usar | ¿Por qué? | Ejemplos |
| :--- | :--- | :--- | :--- |
| **Tiene Senos o Cosenos** ($\sin$, $\cos$) | `True` | Obliga al script a usar la Identidad de Euler para integrar y empaqueta la fracción final sin exponenciales imaginarios. | $e^{-t}\cos(t)$, $\sin(3t)$, $t\cos(t)$ |
| **Solo Exponenciales** ($e^{at}$) | `False` | Mantiene la respuesta limpia. Si lo pones en `True`, intentará forzar senos/cosenos donde no van. | $e^{-\alpha t}$, $e^{-a|t|}$ |
| **Polinomios o Constantes** ($1, t, t^2$) | `False` | Evita que los desplazamientos de tiempo (fase) se transformen en funciones trigonométricas. | $1-t^2$, $5t$, Pulso Unitario |

> **
REGLA DE ORO:** Pon el interruptor en True si tu función original tiene trigonometría O si los límites de integración son simétricos (ej. de $-1$ a $1$, de $-\pi$ a $\pi$, de $-2$ a $2$).

---

## 2. Traductor de Problemas a Tramos (`mis_tramos`)
El script funciona leyendo una lista de intervalos. La estructura base de un tramo es `(funcion, limite_inferior, limite_superior)`.

* **Infinito Negativo:** `-sp.oo`
* **Infinito Positivo:** `sp.oo`

### A. Señales Causales (Empiezan en cero)
El problema dice: $f(t) = \dots$ para $t > 0$, y $0$ para $t < 0$.
```python
mis_tramos = [
    (funcion, 0, sp.oo) 
]
```

### B. Pulsos (Rango cerrado)
El problema dice: $f(t) = \dots$ para $-2 \le t \le 2$.
```python
mis_tramos = [
    (funcion, -2, 2) 
]
```

### C. Señales Bilaterales (Diferente a la izquierda y derecha)
El problema dice: $f(t) = e^{2t}$ para $t < 0$, y $e^{-2t}$ para $t > 0$.
```python
mis_tramos = [
    (sp.exp(2*t), -sp.oo, 0),  # Lado izquierdo
    (sp.exp(-2*t), 0, sp.oo)   # Lado derecho
]
```

---

## 3. Mapa de Herramientas: ¿Qué script debo correr?

Identifica los verbos y pistas en el enunciado del Ing. Saquimuz para decidir qué herramienta abrir en Jupyter Notebook:

* **Enunciado dice:** *"Calcule la transformada por definición"*, *"Encuentre la transformada de $f(t)$"* (y te dan los límites).
    * **USA:** El Script de **fourierportramos** (El que tiene el interruptor de `True/False`).
* **Enunciado dice:** *"Use el teorema de corrimiento"*, *"Aplicando propiedades de traslación"*, o si te piden encontrar una Transformada Inversa $\mathfrak{F}^{-1}$ en base a una tabla.
    * **USA:** El Script del **Aplicador de Teoremas** (El que pide `modulador_w0`). No hay integración aquí, solo álgebra pura.
* **Enunciado dice:** *"Demuestre que..."*, *"Pruebe el teorema..."*
    * **USA:** Tu razonamiento matemático. Las demostraciones (como la de Modulación que vimos) se escriben a mano explicando paso a paso la ley de exponentes.

---

### Tips de Supervivencia (Sintaxis de Python)
* **Siempre usa asterisco para multiplicar:** Nunca escribas `2t` ni `exp(-at)`. Debe ser `2*t` y `sp.exp(-a*t)`.
* **Potencias:** Se usa doble asterisco. $t^3$ es `t**3`.
* **La variable compleja:** Para los problemas del Formulario y Teoremas, si necesitas usar la unidad imaginaria explícita, usa `sp.I` (con i mayúscula).